In [1]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
documents = [file.parse() for file in reader.read()]

In [2]:
!PREFIX=https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main
!wget ${PREFIX}/01-agentic-rag/code/rag_helper.py
!wget ${PREFIX}/04-evaluation/code/evaluation_utils.py

/01-agentic-rag/code/rag_helper.py: Scheme missing.
/04-evaluation/code/evaluation_utils.py: Scheme missing.


In [4]:
!PREFIX=https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main
!wget ${PREFIX}/cohorts/2026/04-evaluation/ground-truth.csv

/cohorts/2026/04-evaluation/ground-truth.csv: Scheme missing.


In [5]:
!wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/cohorts/2026/04-evaluation/ground-truth.csv

--2026-07-13 21:29:30--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/cohorts/2026/04-evaluation/ground-truth.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.111.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 48627 (47K) [text/plain]
Saving to: ‘ground-truth.csv’

ground-truth.csv    100%[===================>]  47.49K  --.-KB/s    in 0.005s  

2026-07-13 21:29:31 (9.44 MB/s) - ‘ground-truth.csv’ saved [48627/48627]



In [6]:
import pandas as pd

# 1. Load the ground truth
df_ground_truth = pd.read_csv('ground-truth.csv')
ground_truth = df_ground_truth.to_dict(orient='records')

# 2. Define the evaluation functions
def hit_rate(relevance_total):
    # If any document matches in top results, it's a hit (1)
    return sum(1 for line in relevance_total if any(line)) / len(relevance_total)

def mrr(relevance_total):
    total_score = 0.0
    for line in relevance_total:
        for rank, val in enumerate(line, start=1):
            if val == 1:
                total_score += 1.0 / rank
                break # Only count the highest rank
    return total_score / len(relevance_total)

def evaluate(search_func, ground_truth_dataset, **kwargs):
    relevance_total = []
    
    for record in ground_truth_dataset:
        query = record['question']
        target_doc = record['filename']
        
        # Execute the search function (e.g., text_search, vector_search, or hybrid_search)
        results = search_func(query, **kwargs)
        
        # Check matching criteria per search result
        # Note: If your chunks match on doc['filename'], use that key
        relevance = [1 if doc.get('filename') == target_doc else 0 for doc in results]
        relevance_total.append(relevance)
        
    return {
        "hit_rate": hit_rate(relevance_total),
        "mrr": mrr(relevance_total)
    }

In [12]:
# 1. Install all missing dependencies needed by the project
!pip install huggingface_hub onnxruntime tokenizers gitsource minsearch

# 2. Download the course's model-fetching helper scripts
!wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/02-vector-search/embed/download.py
!wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/02-vector-search/embed/embedder.py

# 3. Run the script to pull down the Xenova/all-MiniLM-L6-v2 ONNX model files
!python download.py

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 27.1 MB/s  0:00:00m0:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 42.6 MB/s  0:00:006m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.3/35.3 MB 47.3 MB/s  0:00:006m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10/10 [minsearch]10 [pandas]learn]

[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: /usr/local/python/3.12.1/bin/python3 -m pip install --upgrade pip
--2026-07-13 21:40:39--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/02-vector-search/embed/download.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.108.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1376 (1.3K) [text/plain]
Saving to: ‘download.py.1’

download.py.1       100%[===================>]   1.34

In [13]:
!pip install huggingface_hub onnxruntime tokenizers gitsource minsearch
import urllib.request
urllib.request.urlretrieve("https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/02-vector-search/embed/download.py", "download.py")
urllib.request.urlretrieve("https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/02-vector-search/embed/embedder.py", "embedder.py")
!python download.py


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: /usr/local/python/3.12.1/bin/python3 -m pip install --upgrade pip
Traceback (most recent call last):
  File "/workspaces/llm_2026/W4_Evaluation/download.py", line 5, in <module>
    from huggingface_hub import hf_hub_download, list_repo_files
ModuleNotFoundError: No module named 'huggingface_hub'


In [2]:
!pip install onnxruntime


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: /usr/local/python/3.12.1/bin/python3 -m pip install --upgrade pip


In [3]:
pip install sentence-transformers


/workspaces/llm_2026/.venv/bin/python: No module named pip


Note: you may need to restart the kernel to use updated packages.


In [1]:
import numpy as np
import pandas as pd
from gitsource import GithubRepositoryDataReader, chunk_documents
import minsearch

# ==========================================
# WORKAROUND: Mocking the Embedder Class
# ==========================================
# This swaps the ONNX model for standard sentence-transformers
from sentence_transformers import SentenceTransformer

class Embedder:
    def __init__(self):
        # Xenova/all-MiniLM-L6-v2 uses all-MiniLM-L6-v2 under the hood
        self.model = SentenceTransformer('all-MiniLM-L6-v2')
        
    def encode(self, text):
        # ONNX implementation standardizes / normalizes output embeddings
        return self.model.encode(text, normalize_embeddings=True)


# ==========================================
# 1. SETUP DATA AND INITIALIZE INDEXES
# ==========================================
print("Loading data from GitHub and chunking...")
embedder = Embedder()
reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
documents = [file.parse() for file in reader.read()]
chunks = chunk_documents(documents, size=2000, step=1000)

print("Fitting Text Search...")
text_index = minsearch.Index(text_fields=["content"], keyword_fields=[])
text_index.fit(chunks)

print("Fitting Vector Search (Encoding Chunks)...")
embeddings = [embedder.encode(chunk['content']) for chunk in chunks]
X = np.array(embeddings)
vector_index = minsearch.VectorSearch()
vector_index.fit(X, chunks)

# Download and parse Ground Truth
url = "https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/cohorts/2026/04-evaluation/ground-truth.csv"
ground_truth = pd.read_csv(url).to_dict(orient='records')


# ==========================================
# 2. DEFINE SEARCH AND RRF WRAPPERS
# ==========================================
def text_search(query, num_results=5):
    return text_index.search(query, num_results=num_results)

def vector_search(query, num_results=5):
    v_query = embedder.encode(query)
    return vector_index.search(v_query, num_results=num_results)

def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}
    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc
    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

def hybrid_search(query, k=60, num_results=5):
    # Retrieve more candidates internally to give RRF a healthy pool to rank
    text_results = text_search(query, num_results=10)
    vector_results = vector_search(query, num_results=10)
    return rrf([text_results, vector_results], k=k, num_results=num_results)


# ==========================================
# 3. METRICS EVALUATION PARSER
# ==========================================
def hit_rate(relevance_total):
    return sum(1 for line in relevance_total if any(line)) / len(relevance_total)

def mrr(relevance_total):
    total_score = 0.0
    for line in relevance_total:
        for rank, val in enumerate(line, start=1):
            if val == 1:
                total_score += 1.0 / rank
                break
    return total_score / len(relevance_total)

def evaluate(search_func, ground_truth_dataset, **kwargs):
    relevance_total = []
    for record in ground_truth_dataset:
        query = record['question']
        target_filename = record['filename']
        
        results = search_func(query, **kwargs)
        relevance = [1 if doc.get('filename') == target_filename else 0 for doc in results]
        relevance_total.append(relevance)
        
    return {"hit_rate": hit_rate(relevance_total), "mrr": mrr(relevance_total)}


# ==========================================
# 4. RUN ALL EVALUATIONS & DISPLAY ANSWERS
# ==========================================
print("\n=== RUNNING EVALUATIONS ===")

# Q2 & Q3 check
first_query = ground_truth[0]['question']
print(f"\nFirst Query: {first_query}")
print(f"Q2 (Text Search Top Result): {text_search(first_query, num_results=1)[0]['filename']}")
print(f"Q3 (Vector Search Top Result): {vector_search(first_query, num_results=1)[0]['filename']}")

# Q4: Text evaluation
text_metrics = evaluate(text_search, ground_truth, num_results=5)
print(f"\nQ4 (Text MRR): {text_metrics['mrr']:.4f}")

# Q5: Vector evaluation
vector_metrics = evaluate(vector_search, ground_truth, num_results=5)
print(f"Q5 (Vector MRR): {vector_metrics['mrr']:.4f}")

# Q6: Tuning Hybrid Search Loop
print("\nQ6 (Tuning RRF parameter k):")
for k_val in [1, 50, 100, 200]:
    current_hybrid = lambda query, num_results=5: hybrid_search(query, k=k_val, num_results=num_results)
    metrics = evaluate(current_hybrid, ground_truth, num_results=5)
    print(f"  RRF k={k_val:3d} -> Hit Rate: {metrics['hit_rate']:.4f} | MRR: {metrics['mrr']:.4f}")

Loading data from GitHub and chunking...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Fitting Text Search...
Fitting Vector Search (Encoding Chunks)...

=== RUNNING EVALUATIONS ===

First Query: What exactly is a retrieval-augmented generation system, and why does it help with answers that the model wouldn't know on its own?
Q2 (Text Search Top Result): 01-agentic-rag/lessons/03-rag.md
Q3 (Vector Search Top Result): 01-agentic-rag/lessons/01-intro.md

Q4 (Text MRR): 0.5943
Q5 (Vector MRR): 0.6357

Q6 (Tuning RRF parameter k):
  RRF k=  1 -> Hit Rate: 0.8583 | MRR: 0.6723
  RRF k= 50 -> Hit Rate: 0.8472 | MRR: 0.6721
  RRF k=100 -> Hit Rate: 0.8472 | MRR: 0.6721
  RRF k=200 -> Hit Rate: 0.8472 | MRR: 0.6721


In [2]:
import gitsource
import minsearch

print(gitsource.__version__)
print(minsearch.__version__)

0.0.5
0.1.1


In [3]:
print(len(chunks))
print(chunks[0]["filename"], chunks[0]["start"])

295
01-agentic-rag/lessons/01-intro.md 0


In [8]:
# 1. Define the Reciprocal Rank Fusion function
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

# 2. Define the hybrid search function that uses your text and vector tools
def hybrid_search(query, k=60):
    # This assumes you have already created text_search and vector_search functions!
    text_results = text_search(query, num_results=10)
    vector_results = vector_search(query, num_results=10)
    return rrf([text_results, vector_results], k=k)

In [3]:
import sys

# 1. Force Python to install/bootstrap pip into this specific .venv
!{sys.executable} -m ensurepip --default-pip

# 2. Upgrade pip to the latest version to ensure it works smoothly
!{sys.executable} -m pip install --upgrade pip

# 3. Now, install sentence-transformers using the CPU-only low-memory method
!{sys.executable} -m pip install --no-cache-dir torch --index-url https://download.pytorch.org/whl/cpu
!{sys.executable} -m pip install --no-cache-dir transformers sentence-transformers

Looking in links: /tmp/tmp9eqosu55
Processing /tmp/tmp9eqosu55/pip-23.2.1-py3-none-any.whl
  Obtaining dependency information for pip from https://files.pythonhosted.org/packages/5d/95/6b5cb3461ea5673ba0995989746db58eb18b91b54dbf331e72f569540946/pip-26.1.2-py3-none-any.whl.metadata
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 10.3 MB/s eta 0:00:0000:0100:01
  Attempting uninstall: pip
    Found existing installation: pip 23.2.1
    Uninstalling pip-23.2.1:
      Successfully uninstalled pip-23.2.1
Looking in indexes: https://download.pytorch.org/whl/cpu
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 191.8/191.8 MB 246.6 MB/s  0:00:00eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 31.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 110.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 65.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 536.2/536.2 kB 57.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [4]:
import numpy as np
import pandas as pd

from gitsource import GithubRepositoryDataReader, chunk_documents
from gitsource.embedder import Embedder

import minsearch


# ======================================================
# Load repository
# ======================================================

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda p: "/lessons/" in p,
)

documents = [f.parse() for f in reader.read()]
chunks = chunk_documents(documents, size=2000, step=1000)

print("Chunks:", len(chunks))


# ======================================================
# Embedder
# ======================================================

embedder = Embedder()


# ======================================================
# Text index
# ======================================================

text_index = minsearch.Index(
    text_fields=["content"],
    keyword_fields=[]
)

text_index.fit(chunks)


# ======================================================
# Vector index
# ======================================================

X = np.array([embedder.encode(chunk["content"]) for chunk in chunks])

vector_index = minsearch.VectorSearch()
vector_index.fit(X, chunks)


# ======================================================
# Ground truth
# ======================================================

ground_truth = pd.read_csv(
    "https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/cohorts/2026/04-evaluation/ground-truth.csv"
).to_dict(orient="records")


# ======================================================
# Search functions
# ======================================================

def text_search(query, num_results=5):
    return text_index.search(query, num_results=num_results)


def vector_search(query, num_results=5):
    v = embedder.encode(query)
    return vector_index.search(v, num_results=num_results)


def rrf(results, k=60):
    scores = {}
    docs = {}

    for result in results:
        for rank, doc in enumerate(result):
            key = (doc["filename"], doc["start"])

            scores[key] = scores.get(key, 0.0) + 1.0 / (k + rank + 1)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)

    return [docs[k] for k in ranked]


def hybrid_search(query, k=60, num_results=5):
    text = text_search(query, num_results=5)
    vector = vector_search(query, num_results=5)

    return rrf([text, vector], k=k)[:num_results]


# ======================================================
# Metrics
# ======================================================

def evaluate(search_function, **kwargs):

    relevance = []

    for row in ground_truth:

        filename = row["filename"]

        results = search_function(
            row["question"],
            **kwargs
        )

        rel = [
            int(r["filename"] == filename)
            for r in results
        ]

        relevance.append(rel)

    hit_rate = np.mean([any(r) for r in relevance])

    mrr = np.mean([
        next((1/(i+1) for i, x in enumerate(r) if x), 0)
        for r in relevance
    ])

    return hit_rate, mrr


# ======================================================
# Answers
# ======================================================

q = ground_truth[0]["question"]

print()
print("Q2:", text_search(q, 1)[0]["filename"])
print("Q3:", vector_search(q, 1)[0]["filename"])

text_hr, text_mrr = evaluate(text_search)
print("Q4 Hit Rate:", text_hr)
print("Text MRR:", text_mrr)

vec_hr, vec_mrr = evaluate(vector_search)
print("Vector Hit Rate:", vec_hr)
print("Q5 MRR:", vec_mrr)

print()

best_k = None
best_mrr = -1

for k in [1, 50, 100, 200]:

    hr, mrr = evaluate(
        lambda q, num_results=5: hybrid_search(
            q,
            k=k,
            num_results=num_results
        )
    )

    print(f"k={k:3d}  HR={hr:.4f}  MRR={mrr:.4f}")

    if mrr > best_mrr:
        best_mrr = mrr
        best_k = k

print()
print("Q6 best k:", best_k)

ModuleNotFoundError: No module named 'gitsource.embedder'